## Data Retrieval Process

This section covers the process of retrieving the current Dow Jones Industrial Average constituents from Wikipedia and historical price data for each constituent using the yfinance library.

## Retrieving DJIA Constituents and Price Data

First, we fetch the current Dow Jones Industrial Average constituents from Wikipedia and save them to `dow_jones_constituents.csv`. Then, we fetch 10 years of daily adjusted close prices for each constituent using yfinance and save the data to `dow_jones_data.csv`. During this process, we handle any errors by printing the stock symbol that caused the error.

#### AI Prompt:
Write code to perform the following:
Fetch the current Dow Jones Industrial Average constituents from Wikipedia and save them to a CSV file.
Fetch 10 years of daily adjusted close prices for each constituent using yfinance and save the data to a separate CSV file. Handle any errors and print the stock symbol that caused the error.
Load the constituent symbols and price data from the CSV files into a DataFrame, and handle missing data using forward fill.

In [2]:

import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta

# Fetch the Dow Jones Industrial Average constituents from Wikipedia
url = "https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average"
tables = pd.read_html(url)
dow_jones_constituents = tables[1] #The second table on the page contains the required data
dow_jones_constituents.to_csv('dow_jones_constituents.csv', index=False)

# Initialize an empty dataframe for storing stock data
stock_data = pd.DataFrame()

# Set the starting date for historical data (10 years ago from today)
start_date = datetime.now() - timedelta(days=365*10)

for symbol in dow_jones_constituents['Symbol']:
    try:
        ticker_data = yf.download(symbol, start=start_date)
        ticker_data['Symbol'] = symbol
        stock_data = pd.concat([stock_data, ticker_data])
    except Exception as e:
        print(f"Error occurred for symbol: {symbol}, {str(e)}")

# Save the fetched data to CSV
stock_data.to_csv('dow_jones_data.csv')

# Load the data from CSV to pandas DataFrame
dj_constituents = pd.read_csv('dow_jones_constituents.csv')
dj_data = pd.read_csv('dow_jones_data.csv')

# Replace missing data using forward fill method
dj_constituents.fillna(method='ffill', inplace=True)
dj_data.fillna(method='ffill', inplace=True)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%*******

## Implementing the Mean Reversion Strategy

Now that we have the historical price data for the Dow Jones constituents, we can proceed with implementing the mean reversion strategy. The steps involved are:

### Step 1: Calculate Daily Returns

To begin implementing the mean reversion strategy, we first need to calculate the daily returns for each stock in our `data_filled` DataFrame.

#### AI Prompt:

Write Python code to calculate the daily returns (based on the adjusted close prices) for each stock in the DataFrame containing the DJIA prices.

In [3]:
# Sort the DataFrame by 'Symbol' and 'Date'
dj_data.sort_values(['Symbol', 'Date'], inplace=True)

# Calculate daily returns
dj_data['Daily Return'] = dj_data.groupby('Symbol')['Adj Close'].pct_change()

# Preview the DataFrame after calculation
dj_data.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Symbol,Daily Return
10060,2014-08-06,23.687500,23.870001,23.677500,23.740000,20.998867,154232000,AAPL,NaN
10061,2014-08-07,23.732500,23.987499,23.525000,23.620001,20.996647,186844000,AAPL,-0.000106
10062,2014-08-08,23.565001,23.705000,23.320000,23.684999,21.054422,167460000,AAPL,0.002752
10063,2014-08-11,23.817499,24.020000,23.709999,23.997499,21.332214,146340000,AAPL,0.013194
10064,2014-08-12,24.010000,24.219999,23.902500,23.992500,21.327776,135180000,AAPL,-0.000208


### Step 2: Identify Biggest Losers

Next, we will identify the 10 stocks with the lowest returns (biggest losers) for each trading day.

#### AI Prompt:
Write Python code to identify the 10 stocks with the lowest returns for each trading day in the DataFrame containing daily returns. Skip the very first day because that contains only ‘nan’ values. 

In [4]:
# Skip the first date in the dataframe as it contains only 'nan' values
dj_data_without_first_day = dj_data[dj_data['Date'] != dj_data['Date'].min()]

# Group by 'Date' and apply a lambda function to sort and get 10 first items
lowest_daily_returns = dj_data_without_first_day.groupby('Date', as_index=False).apply(lambda x: x.nsmallest(10, 'Daily Return')).reset_index(drop=True)

# View the results
lowest_daily_returns.head(20)

,Date,Open,High,Low,Close,Adj Close,Volume,Symbol,Daily Return
0,2014-08-07,81.000000,81.089996,79.160004,79.260002,68.033737,5465400,UNH,-0.027484
1,2014-08-07,39.959999,40.009998,39.330002,39.349998,28.638943,14367400,KO,-0.014278
2,2014-08-07,86.989998,87.010002,85.230003,85.510002,78.080368,7916900,DIS,-0.012473
3,2014-08-07,81.019997,81.180000,80.010002,80.139999,60.392502,7904000,PG,-0.011715
4,2014-08-07,127.410004,128.889999,125.510002,125.750000,95.151978,3065200,AMGN,-0.010544
5,2014-08-07,38.849998,38.865002,37.990002,38.174999,34.152805,6737400,NKE,-0.010241
6,2014-08-07,49.160000,49.320000,48.450001,48.650002,29.553392,12143700,VZ,-0.009770
7,2014-08-07,178.432129,178.470367,175.506699,176.195023,115.592072,2833196,IBM,-0.008980
8,2014-08-07,52.794998,52.862499,52.150002,52.259998,48.731262,9568000,V,-0.007831
9,2014-08-07,15.760500,15.794000,15.482500,15.572500,15.572500,58712000,AMZN,-0.007773


### Step 3: Simulate Trades

Now, we will simulate buying an equal amount of each of the 10 biggest losers at the close of each trading day and selling all positions at the close of the following trading day. Assume an initial capital of $100,000.

#### AI Prompt:
Code a trading simulation using the DataFrame holding the lowest daily returning stocks:
1.	Ensure dates are in datetime format using pd.to_datetime()
2.	For each date, select the 10 worst performers
3.	Calculate next day's prices using merge with original data, not shift
4.	Start with 100000 dollars, divide equally among 10 stocks daily
5.	Buy at day's adjusted close, sell at next day's adjusted close
6.	Update capital daily
7.	Store results in DataFrame with 'Date' and 'Capital' columns
8.	Skip trades if insufficient capital or missing price data
9.	Handle end of dataset carefully
10.	Print final capital and total return
Use adjusted close prices. Implement robust error checking. Ensure capital isn't erroneously depleted or inflated.

In [9]:
import pandas as pd
import numpy as np

# Ensure dates are in datetime format
lowest_daily_returns['Date'] = pd.to_datetime(lowest_daily_returns['Date'])
dj_data['Date'] = pd.to_datetime(dj_data['Date'])

# Initialize capital and results DataFrame
initial_capital = 100000
results = pd.DataFrame(columns=['Date', 'Capital'])

# Sort data by date
lowest_daily_returns = lowest_daily_returns.sort_values('Date')
dj_data = dj_data.sort_values('Date')

# Get unique dates
unique_dates = lowest_daily_returns['Date'].unique()

for i, current_date in enumerate(unique_dates[:-1]):  # Exclude last date
    next_date = unique_dates[i + 1]
    
    # Select 10 worst performers for the current date
    current_day_losers = lowest_daily_returns[lowest_daily_returns['Date'] == current_date].nsmallest(10, 'Daily Return')
    
    if len(current_day_losers) < 10:
        continue  # Skip if we don't have 10 stocks
    
    # Calculate amount to invest per stock
    amount_per_stock = initial_capital / 10
    
    total_value = 0
    for _, stock in current_day_losers.iterrows():
        symbol = stock['Symbol']
        
        # Find buy and sell prices
        buy_price = stock['Adj Close']
        sell_price = dj_data[(dj_data['Date'] == next_date) & (dj_data['Symbol'] == symbol)]['Adj Close'].values
        
        if len(sell_price) == 0 or np.isnan(buy_price) or np.isnan(sell_price[0]):
            continue  # Skip if we're missing price data
        
        # Calculate number of shares and value
        shares = amount_per_stock / buy_price
        value = shares * sell_price[0]
        
        total_value += value
    
    # Update capital
    initial_capital = total_value if total_value > 0 else initial_capital
    
    # Store results
    new_row = {}
    results.loc[len(results)] = new_row # only use with a RangeIndex!
    results = results.append({'Date': next_date, 'Capital': initial_capital}, ignore_index=True)

# Calculate and print results
final_capital = results['Capital'].iloc[-1]
total_return = (final_capital - 100000) / 100000 * 100

print(f"Final Capital: ${final_capital:.2f}")
print(f"Total Return: {total_return:.2f}%")

AttributeError: 'DataFrame' object has no attribute 'append'

### Step 4: Calculate Performance Metrics

Finally, we will calculate the strategy's annualized return, annualized volatility, Sharpe ratio (assume a risk-free rate of 0), and maximum drawdown.

#### AI Prompt:
Write Python code to calculate the following performance metrics based on the capital each day: Annualized return, Annualized volatility, and Sharpe ratio (assuming a risk-free rate of 0). Then print the calculated metrics.

In [ ]:
# Calculation of Performance Metrics

# Assuming 252 trading days in a year
trading_days = 252

# Compute daily capital returns
results['Capital Return'] = results['Capital'].pct_change() 

# Annualized return
annual_return = (1 + results['Capital Return'].mean())**trading_days - 1

# Annualized volatility
annual_volatility = results['Capital Return'].std() * (trading_days**0.5)

# Sharpe Ratio assuming risk-free rate = 0
sharpe_ratio = annual_return / annual_volatility

# Printing the calculated metrics
print(f"Annualized Return : {annual_return*100:.2f}% ")
print(f"Annualized Volatility : {annual_volatility*100:.2f}% ")
print(f"Sharpe Ratio : {sharpe_ratio:.2f} ")

Annualized Return : 16.04% 
Annualized Volatility : 19.98% 
Sharpe Ratio : 0.80 


### Step 5: Compare with Dow Jones Index

To determine if our mean reversion strategy outperformed the market, we will compare its Sharpe ratio with that of the Dow Jones Index. We'll use the SPDR Dow Jones Industrial Average ETF Trust (DIA) as a proxy for the Dow Jones. The point here is that we want to find out if betting on the losers of the Dow Jones, rather than the Dow Jones itself, is a more profitabl estrategy in hindsight. 

#### AI Prompt:
Write Python code to:
1. Fetch the daily adjusted close prices for the DIA ETF from Yahoo Finance for the same period as the Dow Jones data.
2. Calculate the daily returns for DIA.
3. Calculate the Sharpe ratio for DIA (assume a risk-free rate of 0).
4. Compare the Sharpe ratio of our mean reversion strategy with that of DIA.
5. Print a message indicating whether our strategy outperformed the general Dow Jones based on the Sharpe ratios.

In [ ]:
import numpy as np

# Fetch the DIA ETF data from Yahoo Finance
dia_data = yf.download('DIA', start=start_date)

# Calculate the daily returns for DIA
dia_data['Daily Return'] = dia_data['Adj Close'].pct_change()

# Calculate the Sharpe ratio for DIA
annual_return_dia = (1 + dia_data['Daily Return'].mean())**trading_days - 1
annual_volatility_dia = dia_data['Daily Return'].std() * np.sqrt(trading_days)
sharpe_ratio_dia = annual_return_dia / annual_volatility_dia

print(f"Sharpe Ratio for DIA: {sharpe_ratio_dia:.2f} ")

# Compare Sharpe ratios
if sharpe_ratio > sharpe_ratio_dia:
    print("Our mean reversion strategy outperformed the general Dow Jones.")
else:
    print("Our mean reversion strategy did not outperform the general Dow Jones.")

Sharpe Ratio for DIA: 0.74 
Our mean reversion strategy outperformed the general Dow Jones.


[*********************100%%**********************]  1 of 1 completed


### Step 6: Compare Our Mean Reversion Strategies' Performance to that of the Dow Jones ETF

To better understand the performance of our mean reversion strategy compared to investing in the Dow Jones, we will visualize the annual returns, standard deviations, and Sharpe ratios of both strategies.

#### AI Prompt:
Write Python code to print the mean reversion strategy’s annual return, annual standard deviation, and Sharpe ratio side by side with the DIA. Nicely display which strategy had the higher risk-adjusted returns as measured by the Sharpe ratio.

In [ ]:
# Gathering calculated data for mean reversion strategy and DIA
data = {
    'Mean Reversion Strategy': [annual_return, annual_volatility, sharpe_ratio],
    'DIA': [annual_return_dia, annual_volatility_dia, sharpe_ratio_dia]
}

# Creating a pandas DataFrame with this data
df = pd.DataFrame(data, index=['Annual Return', 'Annual Std Dev', 'Sharpe Ratio'])

# Printing the DataFrame
print(df)

# Finding and printing the strategy with the higher Sharpe Ratio
higher_sharpe_ratio_strategy = df.idxmax(axis=1)['Sharpe Ratio']
print(f"\nStrategy with higher Sharpe Ratio: {higher_sharpe_ratio_strategy}")

                Mean Reversion Strategy       DIA
Annual Return                  0.160399  0.128411
Annual Std Dev                 0.199822  0.174033
Sharpe Ratio                   0.802708  0.737854

Strategy with higher Sharpe Ratio: Mean Reversion Strategy


### Step 7: Visualize Portfolio Growth

To better understand the performance of our mean reversion strategy compared to investing in the Dow Jones, we will visualize the growth of a hypothetical 100,000 dollar portfolio over time for both strategies.

#### AI Prompt:
Write Python code to:
1. Calculate the cumulative returns for both our mean reversion strategy and the DIA ETF.
2. Multiply the cumulative returns by the initial investment of 100,000 dollars to get the daily portfolio values for both strategies.
3. Create a beautiful interactive plot using Plotly to visualize the growth of the $100,000 portfolio over time for both our mean reversion strategy and the DIA ETF.
4. Include proper axis labels, a title, and a legend.
5. Display the plot.

In [ ]:
import plotly.graph_objects as go

# Calculate Cumulative Returns by adding 1 to daily returns, then calculate the cumulative product
results['Cumulative Return'] = (1 + results['Capital Return']).cumprod()
dia_data['Cumulative Return'] = (1 + dia_data['Daily Return']).cumprod()

# Calculate Portfolio Value by multiplying Cumulative Returns by initial capital
results['Portfolio Value'] = results['Cumulative Return'] * 100000
dia_data['Portfolio Value'] = dia_data['Cumulative Return'] * 100000

# Create Line plots 
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=results['Date'], 
    y=results['Portfolio Value'], 
    mode='lines', 
    name='Mean Reversion Strategy'
))

fig.add_trace(go.Scatter(
    x=dia_data.index, 
    y=dia_data['Portfolio Value'], 
    mode='lines', 
    name='DIA ETF'
))

# Add titles and labels
fig.update_layout(
    title='Growth of $100,000 Portfolio Over Time',
    xaxis_title='Date',
    yaxis_title='Portfolio Value ($)',
    legend_title='Strategy',
    autosize=False,
    width=1000, 
    height=500
)

# Display Plot
fig.show()